# dead_drop

> Async file-based IPC: one side `drop()`s a file, another side -- possibly a different process, possibly a different machine reachable only via a synced/shared directory (Dropbox, rsync, an NFS mount, ...) -- `pickup()`s it later, no direct connection between the two required. Named after the spy tradecraft term. This is the "agent-loop file relay" deferred out of `02_remote.ipynb`'s original scope, now built out on its own -- and it subsumes an earlier hand-rolled prototype (`cycle.sh`/`watch.sh`, a bash-only prompt/response relay used to drive a Claude Code session from a Jupyter cell on a different machine) whose good ideas are generalized here: Maildir's atomic-rename trick (`drop()`/`pickup()`), and blocking-wait-on-mtime-change (`watch()`, `next_prompt()`) -- reimplemented in Python rather than shelled out to `stat`, because bash's 1-second mtime resolution left a real race between "checked, nothing pending" and "started waiting for the next change" (found the hard way: a drop landing in that gap could be missed forever).
>
> Deliberately its own flat namespace rather than folded into `slmn.<tool>`: `import slmn.dead_drop as dd` then `dd.drop(...)`, `dd.pickup(...)`, `dd.watch(...)`, `dd.next_prompt(...)`. Everything here blocks or is meant to be called from a blocking loop -- like the previously-deferred `wait.sh`/`launch_queue.sh`-style wrappers elsewhere in slmn -- so this module is import/CLI-only and isn't wired into `cli.py`'s `TOOLS` dict or `mcp.py`'s tool list.

In [ ]:
#| default_exp dead_drop

In [ ]:
#| export
import re, time, uuid
from pathlib import Path
from slmn.nbtools import _resolve_safe

## drop / pickup

One dead drop per message, in a shared directory. `drop()` uses Maildir's write-temp-then-atomic-rename trick, so a concurrent `pickup()`/`watch()` never observes a half-written file.

In [ ]:
#| export
def drop(dir:str, # directory to drop into (created if it doesn't exist)
         content:str, # the drop's body
         name:str=None # filename to use; default is a sortable `{timestamp}-{uuid8}` so pickups see drops in creation order
         ) -> str:
    "Atomically write a dead drop into `dir`, for another process -- possibly on another machine, via a synced/shared directory -- to `pickup()` or `watch()` later. Writes to a temp file in the same directory first, then atomically renames it into place (Maildir's trick), so a concurrent pickup/watch never sees a half-written drop. Returns the dropped file's final path."
    dir_ = Path(_resolve_safe(dir))
    dir_.mkdir(parents=True, exist_ok=True)
    fname = name or f"{time.time_ns()}-{uuid.uuid4().hex[:8]}"
    tmp = dir_ / f".tmp-{uuid.uuid4().hex}"
    tmp.write_text(content)
    final = dir_ / fname
    tmp.rename(final)
    return str(final)

In [ ]:
#| export
def pickup(dir:str, # directory to check for drops
           delete:bool=True # remove the file after reading it, so each drop is picked up exactly once
           ) -> dict: # {'path':..., 'name':..., 'content':...}, or None if nothing's pending
    "Look for the oldest pending drop in `dir` (sorted by filename, so default drop() names -- timestamp-prefixed -- come out in creation order) and read it. Returns None if `dir` doesn't exist or has no drops. Ignores in-progress temp files (`.tmp-*`) so it never races a concurrent drop()."
    dir_ = Path(_resolve_safe(dir))
    if not dir_.exists(): return None
    candidates = sorted(f for f in dir_.iterdir() if f.is_file() and not f.name.startswith('.tmp-'))
    if not candidates: return None
    f = candidates[0]
    content = f.read_text()
    if delete: f.unlink()
    return {'path': str(f), 'name': f.name, 'content': content}

## watch

Blocking loop over `pickup()`. Idles by polling `os.stat().st_mtime` (sub-second precision) instead of re-running `pickup()` on a fixed timer -- a directory's mtime changes when a file is added inside it, so this reacts to a new drop without necessarily waiting out a full `poll_interval`. The mtime baseline is captured *before* the emptiness check, not after, so a drop landing in between is still caught on the very next check rather than silently missed.

In [ ]:
#| export
def _await_change(path:str, # file or directory to watch
                  poll_interval:float, # seconds between mtime checks
                  since:float=None # baseline mtime to compare against; if the path is already newer than this, returns immediately (closes the check-then-wait race -- pass the mtime you observed *before* you checked for existing content, not after)
                  ) -> None:
    "Block until `path`'s mtime is newer than `since` (default: newer than its mtime at call time). Waits for `path` to exist first if it doesn't yet."
    path_ = Path(path)
    while not path_.exists():
        time.sleep(poll_interval)
    base = since if since is not None else path_.stat().st_mtime
    while path_.stat().st_mtime <= base:
        time.sleep(poll_interval)

In [ ]:
#| export
def watch(dir:str, # directory to watch for drops
          callback:callable, # called as callback(drop) for each drop, where drop is the dict pickup() returns
          poll_interval:float=1.0, # seconds between mtime checks while dir is empty
          delete:bool=True, # remove each drop after callback returns (passed through to pickup)
          max_iters:int=None # stop after this many drops-or-idle-checks (mainly for tests/scripted runs -- omit to run forever)
          ) -> int:
    "Blocking loop: repeatedly pickup() drops from `dir` and pass each to `callback`; when `dir` is empty, blocks on a directory-mtime change (see _await_change) instead of busy-polling pickup() itself. Runs forever unless `max_iters` is given -- stop it with Ctrl-C otherwise. Like slmn's other blocking primitives, this is CLI/import-only, never exposed over MCP, since it never returns promptly. Returns the number of drops processed."
    dir_ = Path(_resolve_safe(dir))
    dir_.mkdir(parents=True, exist_ok=True)
    n, iters = 0, 0
    while max_iters is None or iters < max_iters:
        base = dir_.stat().st_mtime  # captured before pickup(), so a drop landing right here is still caught below
        d = pickup(dir, delete=delete)
        if d is not None:
            callback(d)
            n += 1
        else:
            _await_change(str(dir_), poll_interval, since=base)
        iters += 1
    return n

## next_prompt

The other reusable piece of the old prototype: a human types a prompt into a shared file (e.g. from a Jupyter cell on a different machine), possibly multi-line, and marks it submitted with a `---SEND---`-style line so a mid-typing autosave doesn't false-trigger. `cycle.sh` implemented this with an awk block-extraction pass and a `.sentcount` sidecar file; `next_prompt()` is the same idea generalized (any path, any marker regex, no hardcoded filenames) and reimplemented in Python, reusing `_await_change` for the blocking wait.

The response side doesn't need a new primitive -- once a response is composed, `drop()` it to whatever file/directory the other end is watching.

In [ ]:
#| export
def _last_complete_block(path:Path, marker:'re.Pattern'):
    "Scan `path` for lines matching `marker`; return (marker_count, text_of_last_complete_block_or_None) -- the block is everything strictly between the second-to-last and last marker line."
    if not path.exists(): return 0, None
    lines = path.read_text().splitlines()
    idxs = [i for i, l in enumerate(lines) if marker.search(l)]
    if not idxs: return 0, None
    prev = idxs[-2] if len(idxs) >= 2 else -1
    block = '\n'.join(lines[prev + 1:idxs[-1]]).strip()
    return len(idxs), (block or None)

In [ ]:
#| export
def next_prompt(path:str, # path to a growing, marker-delimited prompt log (e.g. a file a human edits in Jupyter/an editor and appends a SEND line to when done typing)
                poll_interval:float=1.0, # seconds between mtime checks
                marker:str=r'^\s*-+\s*send\s*-+\s*$' # case-insensitive regex marking the end of a submitted block; a dash-padded SEND line by default
                ) -> str:
    "Block until `path` gains a new complete marker-delimited block, then return that block's text -- the newest multi-line message the other side submitted. Waits for the file to actually change (see _await_change), then re-checks the marker count, so an unrelated resave or a still-typing autosave doesn't false-trigger. Remembers how many markers it's already consumed in a sidecar state file (`{path}.dd_state`), so repeated calls -- e.g. one per turn of a conversation loop -- never re-fire on old content."
    path_ = Path(_resolve_safe(path))
    marker_re = re.compile(marker, re.IGNORECASE)
    state_file = path_.parent / (path_.name + '.dd_state')
    seen = int(state_file.read_text()) if state_file.exists() else _last_complete_block(path_, marker_re)[0]
    while True:
        base = path_.stat().st_mtime if path_.exists() else 0.0  # captured before the marker re-check, closing the same race watch() closes
        _await_change(str(path_), poll_interval, since=base)
        n, block = _last_complete_block(path_, marker_re)
        if n > seen:
            seen = n
            state_file.write_text(str(n))
            if block: return block
        # else: file changed but no new complete block yet (still typing, or an unrelated resave) -- keep waiting

## Smoke test

In [ ]:
#| eval: false
import tempfile, threading, os

tmp_dir = tempfile.mkdtemp()

# drop / pickup
p1 = drop(tmp_dir, "first message")
p2 = drop(tmp_dir, "second message")
print(p1, p2)

d = pickup(tmp_dir)
assert d['content'] == "first message"
d = pickup(tmp_dir)
assert d['content'] == "second message"
assert pickup(tmp_dir) is None
print("drop/pickup OK")

# watch (drop from a background thread after a short delay -- this exact scenario
# used to hang forever under the old bash-mtime-poll implementation: see module docstring)
received = []
def _delayed_drop():
    time.sleep(0.3)
    drop(tmp_dir, "watched message")
threading.Thread(target=_delayed_drop).start()
n = watch(tmp_dir, received.append, poll_interval=0.1, max_iters=2)
print(n, received)
assert n == 1 and received[0]['content'] == "watched message"
print("watch OK")

# next_prompt
prompt_path = os.path.join(tmp_dir, "prompts.txt")
open(prompt_path, "w").close()

def _delayed_prompt():
    time.sleep(0.3)
    with open(prompt_path, "a") as f:
        f.write("line one\nline two\n---SEND---\n")
threading.Thread(target=_delayed_prompt).start()
prompt = next_prompt(prompt_path, poll_interval=0.1)
print(repr(prompt))
assert prompt == "line one\nline two"
print("next_prompt OK")